#Project 1: Data Collection and Initial Analysis of Stock Market Data
#Name Ayleen Santander

#Important details:
Dataset name: “Daily Historical Stock Prices (1970 - 2020)”
- Date: Current day of transactions
- Open: Open value
- High: Maximun value
- Low: Minimu m value
- Close: Daily close price
- Volume: Daily Volume
- Adjusted Close: Adjusted daily price
You will conduct a detailed exploratory data analysis (EDA) of stock market trends over several decades, focusing on how key financial indicators have evolved over time.

Part 1: Data Collection:

In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import os

In [ ]:
print(os.getcwd())

In [ ]:
#Verify after copy
print([f for f in os.listdir() if f.endswith('.csv')])

In [ ]:
#1. Load the dataset
stock_prices= pd.read_csv('historical_stock_prices.csv', dtype={'volume': 'float64'}, low_memory=False)
hist_stock= pd.read_csv('historical_stocks.csv', dtype={'volume': 'float64'}, low_memory=False)

In [ ]:
#Relevant information about data
print(f'List top 5 of Historical Stock Prices\n',stock_prices.head())
print('--'*45)
print(f'List top 5 of Historical Stocks\n',hist_stock.head())
print('--'*45)

In [ ]:
#Info Historical Stock Prices
print(f'Columns List fo Historical Stock Prices:\n', stock_prices.columns.tolist())
print('--'*45)
col = stock_prices.columns[3]
print('--'*45)
print(stock_prices[col].dtype)
print('--'*45)
print(stock_prices[col].unique()[:20])

In [ ]:
#2. Handle any missing values, outliers, and preprocess the data (e.g., normalization, encoding categorical variables).
stock_prices.info()
print(f'Missing values in Historical Stock Prices:\n',stock_prices.isnull().sum())
stock_prices.shape
stock_prices.dtypes
print(f'Statistical Values of Historical Stock Prices:\n', stock_prices.describe().T )

#Part 2: Data Cleaning

In [ ]:
#1. Identify and handle missing values in the datasets. Choose a strategy to deal with them (e.g., filling with the mean or median, interpolation, or removal).
# Null check
nulls = stock_prices.isnull().sum()
print("Nulls per column:")
print(nulls)
print(f"\nTotal nulls: {nulls.sum()}")

print('=='* 45)
# Duplicates are worth checking too
print(f"Duplicate rows: {stock_prices.duplicated().sum()}")

print('=='* 45)
# Overall structure
print(type(stock_prices))

In [ ]:
#Check the nulls values
bad = stock_prices[stock_prices.isna().any(axis=1)]
print(bad)

In [ ]:
#Drop it the nan
stock_prices = stock_prices.dropna()

In [ ]:
#Check duplicate values
print(stock_prices.duplicated(subset=['ticker','date']).sum())

In [ ]:
#Drop because is the same value, and keep the first
stock_prices = stock_prices.drop_duplicates(subset=['ticker','date'], keep='first')

In [ ]:
#After changes and remove duplicated values
nulls = stock_prices.isnull().sum()
print("Nulls per column:")
print(nulls)
print(f"\nTotal nulls: {nulls.sum()}")

print('=='* 45)
# Duplicates are worth checking too
print(f"Duplicate rows after drop them: {stock_prices.duplicated().sum()}")

In [ ]:
#Missing values in the STOCKS dataset (you only checked prices)
print(hist_stock.isnull().sum())

In [ ]:
#Fix te date column and adj_close
stock_prices['date'] = pd.to_datetime(stock_prices['date'], errors='coerce')
stock_prices = stock_prices.sort_values(['ticker','date']).reset_index(drop=True)
print(stock_prices.sort_values(['ticker', 'date']))

print('--'*45)

stock_prices['adj_close'] = pd.to_numeric(stock_prices['adj_close'], errors='coerce')
print(stock_prices['adj_close'].isna().sum())

In [ ]:
stock_prices = stock_prices.set_index('date').sort_index()
print(stock_prices.dtypes)
print(stock_prices.head())

In [ ]:
#Group by monthly each ticket
monthly = stock_prices.groupby('ticker')['close'].resample('ME').mean()
print(monthly)

#Part 3: Data Segmentation by Decade

In [ ]:
#1. Create a new column in the DataFrame for prices to represent the decade.
stock_prices['decade'] = (stock_prices.index.year // 10) * 10

print(stock_prices['decade'].value_counts().sort_index())
print('--'*45)
print(stock_prices.head())

In [ ]:
#Lables instead of numbers
stock_prices['decade'] = (stock_prices.index.year // 10 * 10).astype(str) + 's'
print(stock_prices['decade'].value_counts().sort_index())
print('--'*45)
print(stock_prices)

In [ ]:
#2. Segment the data into separate DataFrames for each decade for easier comparative analysis.
#Dictionary keyed by decade
decades= {d: g for d, g in stock_prices.groupby('decade')}

for d, df in decades.items():
    print(f'Decade: {d}')
    print(df.head())
    print('--'*45)

In [ ]:
#Fix= drop the bad row (nans)
# remove the row with no date
stock_prices = stock_prices[stock_prices.index.notna()]

# recompute decade, now clean integers
stock_prices['decade'] = (stock_prices.index.year // 10) * 10

print(stock_prices['decade'].value_counts().sort_index())

In [ ]:
#Rebuild the segments after cleaned
decades = {d: g for d, g in stock_prices.groupby('decade')}
for d, df in decades.items():
    print(f"{d}s — {df.shape[0]:,} rows, {df['ticker'].nunique()} tickers")

In [ ]:
#Check by decades
for d, df in decades.items():
    print(f"{d}s — mean close: {df['close'].mean():.2f}, tickers: {df['ticker'].nunique()}")

##Part 4: Exploratory Data Analysis by Decade

In [ ]:
#1. Merge the stock_prices dataset with the stock dataset using the appropriate key to include sector
#information with the stock prices.
#inspect the other dataset
print(f'Columns List:', hist_stock.columns.tolist())
print(f'Shape Rows and Columns:', hist_stock.shape)
print(f'Duplicated values:', hist_stock['ticker'].duplicated().sum())

In [ ]:
#Merge both dataset with pd.merge
merged = (
    stock_prices.reset_index()
    .merge(hist_stock[['ticker', 'sector']], on='ticker', how='left')
    .set_index('date')
)

#check coverage
print(merged['sector'].isna().sum())            # tickers with no sector match
print(merged[merged['sector'].isna()]['ticker'].unique()[:20])
print(merged.shape[0], stock_prices.shape[0])   # must match — no row inflation

In [ ]:
#Convert to save space:
merged['sector'] = merged['sector'].astype('category')

In [ ]:
#2. For each decade DataFrame, calculate summary statistics (mean, median, standard deviation)
#for the Open, High, Low, Close, and Volume columns. Use .agg()
cols =['open', 'high', 'low', 'close', 'volume']
for d, df in decades.items():
  print(f'\n==={d}s===')
  print(df[cols].agg(['mean', 'median', 'std']).round(2))

In [ ]:
#Additional all in one table
summary = stock_prices.groupby('decade')[cols].agg(['mean', 'median', 'std']).round(2)
print(summary)

In [ ]:
#3. Create visualizations for each decade:
# Time series plots for average monthly Close prices.
ordered = sorted(decades.items())

fig = make_subplots(
    rows=len(ordered), cols=1,
    subplot_titles=[f'{d}s' for d, _ in ordered],
    vertical_spacing=0.06
)

for i, (d, df) in enumerate(ordered, start=1):
    monthly = df['close'].resample('ME').mean()
    fig.add_trace(
        go.Scatter(x=monthly.index, y=monthly.values,
                   mode='lines', name=f'{d}s'),
        row=i, col=1
    )
    fig.update_yaxes(title_text='Avg Close ($)', row=i, col=1)

fig.update_layout(
    height=300 * len(ordered),
    title='Average Monthly Close Price by Decade',
    showlegend=False,
    template='plotly_white'
)
fig.show()

Explanation:
1970s: Peaks at $7.60 in May 1972, falls to a trough of $3.39 in December 1974,  a 55% decline. Recovers to $5.94 by December 1979 without regaining the 1972 level.

This matches the 1973–74 bear market closely. It's credible because the sample is just 93 large, stable tickers with little composition churn.

1980s: Underlying drift from ~$6 to ~$23 is consistent with the bull market beginning August 1982. But isolated one-month spikes interrupt it:

Jun 1981 ($20.96) · Sep 1982 ($31.04) · Nov 1982 ($20.11) · Feb 1983 ($23.95) · Jul 1983 ($32.16) · Jan 1986 ($40.68) · Apr 1986 ($45.08) · Feb–Apr 1987 · Aug 1987 ($58.17)

A cross-sectional mean that triples and reverts within one month is a composition artifact. Note October 1987 shows nothing, $22.39, down modestly from $24.67. The crash is absent while fictitious spikes are present.

1990s: Declines from $24 to $15–16 through 1994–95, then trends up to $40.00 by December 1999. Spikes at Mar 1998 ($92.86), Feb 1998 ($67.50) and May 1998 ($64.08) break a baseline near $30. The late-decade rise is plausibly the dot-com run-up, but composition churn makes it impossible to separate from sampling change.

2000s: Feb 2000 = $460.32 and Mar 2000 = $309.56, against $65 and $70 either side. Further spikes at Jun 2000 ($211.88) and Sep 2000 ($192.00).

From 2003 the series stabilizes: rises to $378.16 in Mar 2005, declines through 2006–07, then the October 2008 fall from $117.22 to $57.78, a 51% single-month drop matching the financial crisis. Flat $38–56 through 2009.

2010s: Baseline $45–65, with sustained excursions to $192.08 (Sep 2013) and $179.63 (Oct 2013), then $128–136 in early 2014, returning to the low $40s by 2016. The final months oscillate violently, roughly $46, $97, $42, $99, $97. Thousands of stocks cannot collectively halve and double in alternating months.

This is an unweighted cross-sectional mean over a changing ticker universe. Each month averages whichever stocks were listed, so the line moves when composition changes, not only when prices do. With thousands of tickers this should average out, that it doesn't confirms extreme values remain.

Conclusion: Two features across the full 1970–2018 span are genuinely interpretable: the 1973–74 decline and the October 2008 crash. Everything else reflects data composition rather than market behaviour. Identifying that is a stronger finding than narrating spikes as events.


In [ ]:
#Definitions of clean prices
clean_prices = stock_prices[stock_prices['adj_close'].between(0, 100_000)]
print(f"Removed {len(stock_prices) - len(clean_prices):,} rows")

In [ ]:
#Clean prices
monthly = clean_prices.groupby('decade')['close'].resample('ME').median()

# rebuild decade segments from the CLEANED frame
decades_clean = {d: g for d, g in clean_prices.groupby('decade')}
ordered = sorted(decades_clean.items())

fig = make_subplots(rows=len(ordered), cols=1, subplot_titles=[f'{d}s' for d, _ in ordered], vertical_spacing=0.06)

for i, (d, df) in enumerate(ordered, start=1):
    monthly = df['close'].resample('ME').median()
    fig.add_trace(
        go.Scatter(x=monthly.index, y=monthly.values, mode='lines'),
        row=i, col=1
    )
    fig.update_yaxes(title_text='Median Close ($)', row=i, col=1)

fig.update_layout(height=300 * len(ordered), title='Median Monthly Clean Close Price by Decade', showlegend=False, template='plotly_white')
fig.show()

Explanation:
Every spike is gone. Y-axes now read $2–24 across all five panels instead of $460. The median did what the mean couldn't.

1970s= $ 3.67 → $4.34. Clear bear market: peak $5.08 (Jan 1973) falling to trough $1.88 (Oct–Nov 1974), a 63% decline. Recovers to ~$4.50 by 1979. Textbook 1973–74.

1980s= $4.72 → $7.05. Dip to $2.08 (Apr 1980), then a steady climb from 1982 onward. Peak $6.72 (Aug 1987), then a drop to $4.75 by Oct 1987, a 29% fall. The 1987 crash is now visible; it was completely hidden in the mean.

1990s = $7.00 → $12.73, the smoothest series in the set. Steady rise to $14.88 (Apr 1998), then a decline through late 1998 and a flat $12.50–13.90 through 1999. Note the typical stock does not participate in the dot-com run-up,that was concentrated in a small number of names.

2000s = $13.00 → $16.08. Flat through 2002, rising to $23.93 (Apr 2007), then falling to $10.02 (Feb 2009), a 58% decline. The crash now reads as a gradual 2008 slide rather than a single October cliff, which is what actually happened.

2010s= $16.59 → $22.76. Steady climb to ~$23.90 (Mar 2014), a dip to $17.62 (Feb 2016), then recovery to ~$22.80.

The median series recovers four market events (1973–74, 1987, 2000–02, 2007–09) where the mean showed only two, and none of the artificial spikes. Median-of-cross-section is the correct statistic for a panel with a changing, outlier-prone ticker universe.

In [ ]:
#Single continuous char with a range slider
monthly_all = clean_prices['close'].resample('ME').median().reset_index()
monthly_all.columns = ['date', 'median_close']

fig = px.line(monthly_all, x='date', y='median_close',
              title='Median Monthly Close Price, 1970–2018',
              log_y=True, template='plotly_white')
fig.update_xaxes(rangeslider_visible=True)
fig.show()

Explanation:
Median Monthly Close, 1970–2018 (log scale): Clean data, robust statistic, log axis, all three fixes applied. Range $1.88 to $23.94, no spikes.

The long-run trend: $3.67 (Jan 1970) → $22.76 (Aug 2018), roughly 6× in nominal terms over 48 years. On a log axis that's a fairly straight upward line, meaning steady proportional growth rather than acceleration.

The 2007–09 drawdown is the deepest of the period in percentage terms except 1973–74, and the typical stock took until roughly 2014 to regain its 2007 level, a seven-year recovery.

The dot-com bust barely registers for the median stock (−23%) despite being a defining market event. That's because the bubble was concentrated in a narrow set of tech names; the median listed company never participated, so it had little to give back.
Note: nominal dollars, no inflation adjustment; survivorship bias since only tickers listed as of 2018 appear; 2,705 rows excluded as data errors.

In [ ]:
#Histograms and boxplots of close
samp = stock_prices.sample(500_000, random_state=42)
px.histogram(samp, x='close', nbins=100, log_y=True).show()
px.box(samp, y='close').show()

In [ ]:
#One stock over time
one = stock_prices[stock_prices['ticker'] == 'KO'].sort_index()
px.line(one, y='close', title='KO closing price, 1970–2018').show()

In [ ]:
# #Export the portfolio
# fig.write_html('monthly_close.html')     # interactive, works on GitHub Pages
# fig.write_image('monthly_close.png')     # static; needs: pip install -U kaleido

In [ ]:
#Histograms for Volume to analyze the distribution and any shifts over the decades.
plot_df = stock_prices[stock_prices['volume'] > 0].copy()
plot_df['log_volume'] = np.log10(plot_df['volume'])

fig = px.histogram(
    plot_df, x='log_volume', facet_row='decade',
    nbins=80, height=250 * plot_df['decade'].nunique(),
    title='Distribution of Daily Volume by Decade (log scale)',
    labels={'log_volume': 'log₁₀(Volume)'},
    template='plotly_white'
)
fig.update_yaxes(matches=None)   # let each decade use its own y-scale
fig.show()

In [ ]:
#Overlaid version
fig = px.histogram(
    plot_df, x='log_volume', color='decade',
    nbins=100, barmode='overlay', opacity=0.55,
    histnorm='probability density',
    title='Volume Distribution Shift Across Decades',
    labels={'log_volume': 'log₁₀(Volume)'},
    template='plotly_white'
)
fig.show()

In [ ]:
#Memory warning
plot_df = plot_df.sample(500_000, random_state=42)

In [ ]:
#Box plots for the High and Low prices to examine the range and presence of outliers.
#Visual Box plot
px.box(plot_df, x='decade', y='log_volume', template='plotly_white', color_discrete_sequence=['purple']).show()

In [ ]:
#Part 5: Comparative Analysis

In [ ]:
#Compare the summary statistics across decades and document any notable trends or changes in stock price
#behaviors and trading volumes.
cols = ['open', 'high', 'low', 'adj_close', 'volume']

comparison = stock_prices.groupby('decade').agg(
    n_rows=('adj_close', 'size'),
    n_tickers=('ticker', 'nunique'),
    close_mean=('adj_close', 'mean'),
    close_median=('adj_close', 'median'),
    close_std=('adj_close', 'std'),
    vol_mean=('volume', 'mean'),
    vol_median=('volume', 'median'),
    vol_std=('volume', 'std')
).round(2)

comparison['close_cv'] = (comparison['close_std'] / comparison['close_mean']).round(2)
comparison['vol_skew_ratio'] = (comparison['vol_mean'] / comparison['vol_median']).round(2)

print(comparison)

In [ ]:
print(stock_prices.nlargest(20, 'adj_close')[['ticker', 'adj_close', 'close', 'volume']])

In [ ]:
print((stock_prices['adj_close'] > 100_000).sum())
print((stock_prices['adj_close'] < 0).sum())
print(stock_prices['adj_close'].describe())

In [ ]:
# Header-like ticker records
print(hist_stock[hist_stock['ticker'].str.contains('ticker', case=False, na=False)])
print(stock_prices[stock_prices['ticker'].str.len() > 6]['ticker'].unique()[:20])

In [ ]:
bad = stock_prices[stock_prices['adj_close'] > 10_000]
print(bad[['ticker', 'close', 'adj_close']].head(20))

In [ ]:
clean_prices = stock_prices[
    (stock_prices['adj_close'] > 0) &
    (stock_prices['adj_close'] < 100_000)
]
print(f"Removed {len(stock_prices) - len(clean_prices):,} rows")

In [ ]:
#Clear comparison, restrict tickeets present in all 5 decades.
per_decade = stock_prices.groupby('ticker')['decade'].nunique()
survivors = per_decade[per_decade == 5].index
print(len(survivors))

In [ ]:
#New Summary from the cleaned frame
comparison = clean_prices.groupby('decade').agg(
    n_rows=('adj_close', 'size'),
    n_tickers=('ticker', 'nunique'),
    close_mean=('adj_close', 'mean'),
    close_median=('adj_close', 'median'),
    close_std=('adj_close', 'std'),
    vol_mean=('volume', 'mean'),
    vol_median=('volume', 'median'),
    vol_std=('volume', 'std')
).round(2)

comparison['close_cv'] = (comparison['close_std'] / comparison['close_mean']).round(2)
comparison['vol_skew_ratio'] = (comparison['vol_mean'] / comparison['vol_median']).round(2)

print(comparison)

Explanation:
Rows rise from 164,499 (93 tickers) in the 1970s to 6,190,902 (3,768 tickers) in the 2010s, a 57× increase in observations and 61× in tickers. Each decade describes a different population of companies, so every comparison below is between samples rather than a fixed panel tracked over time.

The median climbs monotonically, $0.40 to $18.19, a 45× increase. The mean now rises alongside it through the 2000s before easing slightly in the 2010s, which is consistent rather than contradictory. Because these are split- and dividend-adjusted prices, early-decade values are expressed in terms of what a share was worth relative to today, which is why 1970s figures sit well below a dollar.

The coefficient of variation peaks at 24.00 in the 1980s, then declines steadily: 17.93 → 19.57 → 12.52. The 2010s figure is roughly half the 1980s. Raw standard deviation rises because prices are larger, but relative to the mean, the spread between cheap and expensive stocks has narrowed. The 1970s CV of 1.71 is an outlier driven by the tiny 61-ticker sample of large, homogeneous firms.

vol_median is essentially flat and slightly declines: 183,600 (1970s) → 162,300 (2010s)
vol_std rises 18×: 1.2M → 21.5M

This is the most striking finding in the table. The typical stock does not trade more heavily today than in 1970. What changed is the spread, the gap between the busiest and quietest names widened enormously as thousands of thinly traded small-caps entered the market alongside a handful of dominant mega-caps.

Mean/median ratio: 3.51 → 13.11 → 13.77 → 10.24 → 8.34. The low 1970s value reflects a sample of only 56 large, liquid firms with similar turnover. The rise through the 1990s tracks the entry of thinly traded issues. The post-2000 decline suggests liquidity has broadened across more names rather than concentrating further.

Price dispersion and volume dispersion move in opposite directions after 2000. Price CV falls from 19.57 to 12.52 while volume std nearly doubles from 10.6M to 21.5M. Stocks have become more similar in price and less similar in how actively they trade.

Limitations:

1. Survivorship bias. Only tickers listed as of 2018 appear. Delisted and bankrupt firms are absent, biasing any performance inference upward. This is the most significant constraint on the analysis.
2. Changing composition. Decade differences confound market evolution with sampling change; the ticker universe is not held constant.
3. Nominal dollars. No inflation adjustment applied. A 1970 dollar is worth roughly 6–7× a 2018 dollar.
4. Removed observations. 2,705 rows were excluded as data errors. This is documented but represents a judgment call.

In [ ]:
#Exchanges and sectors
px.bar(hist_stock['exchange'].value_counts(), title='Securities per Exchange').show()
px.bar(hist_stock['sector'].value_counts(), title='Securities per Sector').show()
print('Missing sector:', hist_stock['sector'].isna().sum())

In [ ]:
#Highest CLOSE (you only did adj_close)
print(stock_prices.nlargest(15, 'close').reset_index()[
    ['ticker','date','open','high','low','close','volume']])

In [ ]:
#Define decades_clean
for name in ['decades', 'decades_clean']:
    if name in globals():
        del globals()[name]
gc.collect()

In [ ]:
#light version
mask = ~stock_prices['adj_close'].between(0, 100_000)
print(stock_prices.loc[mask, 'decade'].value_counts().sort_index())
print(stock_prices.loc[mask, 'ticker'].value_counts().head(10))